# Chapter 32: Fine-Tuning Models for Network Data

**What You Will Learn:**
- The key question: should you fine-tune or use RAG? (and how to decide)
- What fine-tuning actually does to the model weights — simply explained
- What LoRA is and why it makes fine-tuning practical (with pure Python math)
- How to build a high-quality JSONL training dataset from your network data
- How to generate synthetic training examples using a teacher LLM
- How to evaluate whether fine-tuning actually improved anything

**The core insight from this chapter:**

> "Fine-tuning is for **form**. RAG is for **facts**."

Fine-tuning teaches the model *how* to respond — format, style, tone, domain language.
RAG gives the model *what* to respond with — current facts, your private data.

Most teams reach for fine-tuning too early.
This chapter teaches you when it actually makes sense — and how to do it right.

**What about the actual training code?**

Training requires a GPU with 16-40GB VRAM + HuggingFace libraries.
In this notebook:
- **All demos run in Colab** — dataset prep, evaluation, synthetic data generation
- **Training loop is shown as reference** — explains every line, but needs GPU to execute
- **You get the complete pipeline** — ready to run on Google Colab Pro / Vast.ai / Modal

**Prerequisites:** Anthropic API key, Python 3.8+

In [ ]:
# Setup — core libraries only (training libs shown separately)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "anthropic", "-q"])

import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')

from anthropic import Anthropic
import json, random, math, time, re
from collections import defaultdict

client = Anthropic()
MODEL = "claude-haiku-4-5-20251001"

random.seed(42)
print("✓ Setup complete")
print(f"  Model: {MODEL}")
print()
print("Note: Demos 1-4 run fully in Colab.")
print("      Demo 5 (training loop) needs a GPU environment.")
print("      All code is production-ready — just add GPU and run.")

## Demo 1: Should You Fine-Tune? — The Decision Framework

Before writing a single line of training code, ask these questions.
Getting this wrong is expensive — fine-tuning takes days and costs money.

**The Golden Rule:**
```
Fine-tuning  =  teach the model HOW to respond  (form, style, format)
RAG          =  give the model WHAT to say       (facts, current data)
```

**When fine-tuning WINS:**
- Your domain has unique terminology not in general training data
- You need a specific output FORMAT every time (JSON schema, CLI syntax)
- You want the model to "sound like" your team's documentation style
- You're repeating 2,000+ tokens of context in every prompt (baked-in context)

**When RAG WINS:**
- You need up-to-date information (network topology changes weekly)
- You want to audit what information the model used
- Your "knowledge" is really just a document library you query
- You need to add/remove knowledge without retraining

**When you need BOTH:**
- Fine-tune for format + domain style
- RAG for current device states, tickets, configurations


In [ ]:
# Demo 1: Interactive Fine-Tuning vs RAG Decision Helper

class FineTuningDecisionHelper:
    """
    Asks key questions and recommends fine-tuning vs RAG vs both.
    Based on the decision framework from the book.
    """

    QUESTIONS = [
        {
            "id": "format",
            "question": "Do you need a very specific output format every time?",
            "hint": "(e.g., always JSON, always CLI commands, always a table)",
            "weight_ft": 2,  # fine-tuning score
            "weight_rag": 0,
        },
        {
            "id": "context",
            "question": "Are you repeating >1000 tokens of context in every prompt?",
            "hint": "(e.g., device inventory, naming conventions, OSPF area design)",
            "weight_ft": 3,
            "weight_rag": 1,
        },
        {
            "id": "updates",
            "question": "Does your knowledge change frequently? (weekly/monthly?)",
            "hint": "(e.g., routing tables, device inventory, ticket history)",
            "weight_ft": -2,  # negative = RAG is better
            "weight_rag": 3,
        },
        {
            "id": "private_docs",
            "question": "Do you have a large library of private documents to query?",
            "hint": "(e.g., runbooks, config archives, NOC procedures)",
            "weight_ft": 0,
            "weight_rag": 3,
        },
        {
            "id": "audit",
            "question": "Do you need to trace WHICH document the model used?",
            "hint": "(e.g., compliance requires knowing the source of advice)",
            "weight_ft": -1,
            "weight_rag": 3,
        },
        {
            "id": "terminology",
            "question": "Does your team use proprietary terminology the model doesn't know?",
            "hint": "(e.g., internal device names, custom alert codes, vendor-specific syntax)",
            "weight_ft": 2,
            "weight_rag": 1,
        },
        {
            "id": "small_data",
            "question": "Do you have <500 high-quality labeled examples?",
            "hint": "(fine-tuning needs quality examples — <50 is usually not enough)",
            "weight_ft": -1,
            "weight_rag": 2,
        },
    ]

    def evaluate(self, answers: dict) -> dict:
        """
        answers: dict of {question_id: True/False}
        Returns recommendation with reasoning.
        """
        ft_score  = 0
        rag_score = 0
        reasoning = []

        for q in self.QUESTIONS:
            ans = answers.get(q["id"], False)
            if ans:
                ft_score  += q["weight_ft"]
                rag_score += q["weight_rag"]
                reasoning.append(f"  [+] {q['question']}")

        # Recommendation logic
        if ft_score >= 4 and rag_score >= 4:
            rec     = "BOTH (Fine-tune + RAG)"
            detail  = "Fine-tune for format/style, use RAG for dynamic knowledge"
        elif ft_score > rag_score and ft_score >= 3:
            rec     = "FINE-TUNING"
            detail  = "Knowledge is stable, format consistency matters"
        elif rag_score > ft_score and rag_score >= 3:
            rec     = "RAG"
            detail  = "Knowledge changes often, documents are primary source"
        else:
            rec     = "PROMPT ENGINEERING FIRST"
            detail  = "Not enough evidence for either — try prompt engineering first"

        return {"recommendation": rec, "detail": detail,
                "ft_score": ft_score, "rag_score": rag_score,
                "reasoning": reasoning}

    def print_result(self, result: dict):
        print("\n" + "="*55)
        print("Fine-Tuning vs RAG Decision")
        print("="*55)
        print(f"  Fine-Tuning score: {result['ft_score']}")
        print(f"  RAG score:         {result['rag_score']}")
        print()
        print(f"  → RECOMMENDATION: {result['recommendation']}")
        print(f"    {result['detail']}")
        print()
        if result["reasoning"]:
            print("  Key factors that drove this decision:")
            for r in result["reasoning"]:
                print(r)
        print("="*55)


# ── Test with a realistic network AI scenario ──────────────────────────────

helper = FineTuningDecisionHelper()

# Scenario 1: NOC alert classifier
print("Scenario 1: NOC Alert Classifier")
print("  - Want JSON output format every time")
print("  - Repeating device inventory in every prompt (2000 tokens)")
print("  - Alert patterns are mostly stable (BGP/OSPF/interface)")
print("  - 200 labeled historical alerts available")
answers_1 = {
    "format":      True,   # need JSON output
    "context":     True,   # repeat inventory
    "updates":     False,  # patterns stable
    "private_docs": False,
    "audit":       False,
    "terminology": True,   # internal alert codes
    "small_data":  False,  # 200 examples — enough
}
result_1 = helper.evaluate(answers_1)
helper.print_result(result_1)

print()

# Scenario 2: NOC runbook lookup
print("Scenario 2: NOC Runbook Question-Answering")
print("  - Engineers ask questions about procedures")
print("  - 500+ runbooks that update quarterly")
print("  - Need to cite which runbook was used")
answers_2 = {
    "format":      False,
    "context":     False,
    "updates":     True,   # runbooks update
    "private_docs": True,  # 500+ runbooks
    "audit":       True,   # need source citation
    "terminology": True,   # some internal terms
    "small_data":  True,   # not enough labeled Q&A pairs
}
result_2 = helper.evaluate(answers_2)
helper.print_result(result_2)

## Demo 2: What Fine-Tuning Does — Weight Updates Visualized

Fine-tuning updates the model's weights using your training examples.

**The key difference:**

| Method | What changes | Memory |
|---|---|---|
| Prompt engineering | Nothing — weights stay the same | Only in current context |
| RAG | Nothing — weights stay the same | Only in current context |
| Fine-tuning | The weights ARE changed | Baked into the model permanently |

**LoRA: the smart way to fine-tune**

Full fine-tuning changes ALL weights. A 7B model has 7 billion numbers to update.
LoRA only updates a tiny fraction — by adding small "adapter" matrices.

```
Full weights W (4096 × 4096 = 16.7M numbers)
LoRA update:  B × A   where A = (4096×8), B = (8×4096) = 65,536 numbers
Reduction:    256x fewer parameters to update!
```

**The LoRA rank `r`:**
- r=4:  very small adapters, fastest training, good for simple style changes
- r=8:  standard, works for most domain adaptation tasks
- r=16: larger, for complex domain shifts
- r=64: approaching full fine-tuning territory

**Networking analogy (from the book):**
> "In BGP, when a route changes, you don't re-advertise your entire routing table —
> you send an UPDATE with just the delta.
> LoRA is the same: instead of updating all 16 billion weights,
> you compute just the meaningful change (the delta B×A).
> The original routing table (base model) stays intact." 

In [ ]:
# Demo 2: LoRA Concept Visualized with Pure Python Math
# No PyTorch needed — just numpy-free matrix multiplication

def mat_mul(A, B):
    """Multiply two matrices. A: (m×k), B: (k×n) → result (m×n)."""
    m, k = len(A), len(A[0])
    k2, n = len(B), len(B[0])
    assert k == k2, f"Matrix dimensions mismatch: {k} vs {k2}"
    return [[sum(A[i][j] * B[j][l] for j in range(k))
             for l in range(n)] for i in range(m)]

def mat_add(A, B):
    """Add two matrices element-wise."""
    return [[A[i][j] + B[i][j] for j in range(len(A[0]))]
            for i in range(len(A))]

def count_params(matrix):
    """Count total parameters in a matrix."""
    return len(matrix) * len(matrix[0])

def format_matrix(M, name, max_rows=4):
    """Pretty print a matrix (truncated)."""
    rows = len(M)
    cols = len(M[0])
    print(f"  {name} ({rows}×{cols} = {rows*cols:,} params):")
    for i, row in enumerate(M[:max_rows]):
        vals = " ".join(f"{v:>6.3f}" for v in row[:6])
        suffix = "  ..." if cols > 6 else ""
        print(f"    [{vals}{suffix}]")
    if rows > max_rows:
        print(f"    ... ({rows - max_rows} more rows)")


# ── Simulate a tiny "weight matrix" from a neural network layer ─────────────

import random
random.seed(7)

# Simulate a weight matrix W (real models: 4096×4096, we use 8×8 for demo)
DIM   = 8     # model dimension (real: 4096)
RANK  = 2     # LoRA rank r (real: 4-16)

# Original pre-trained weights (randomly initialized for demo)
W_original = [[random.gauss(0, 0.1) for _ in range(DIM)] for _ in range(DIM)]

print("LoRA Concept Demo — tiny 8×8 matrices (real models: 4096×4096)\n")
print("─" * 55)

# ── Full fine-tuning: would update ALL weights ─────────────────────────────
full_params = count_params(W_original)
print(f"Full fine-tuning would update: {full_params:,} parameters")
print(f"  Real model (4096×4096):       {4096*4096:,} parameters per layer")

# ── LoRA: only updates A (DIM×RANK) and B (RANK×DIM) ─────────────────────
# Initialize A with small random values, B with zeros (standard LoRA init)
A = [[random.gauss(0, 0.02) for _ in range(RANK)] for _ in range(DIM)]
B = [[0.0 for _ in range(DIM)] for _ in range(RANK)]

lora_params = count_params(A) + count_params(B)
reduction   = full_params / lora_params

print(f"\nLoRA with rank r={RANK}:")
print(f"  A matrix ({DIM}×{RANK}): {count_params(A):,} parameters")
print(f"  B matrix ({RANK}×{DIM}): {count_params(B):,} parameters")
print(f"  Total LoRA params:   {lora_params:,} parameters")
print(f"  Reduction:           {reduction:.0f}x fewer parameters!")
print(f"\n  Real model (4096×4096, r=8):")
print(f"    LoRA params: {2*4096*8:,} vs full: {4096*4096:,} = {4096*4096//(2*4096*8)}x smaller")

# ── Simulate forward pass with LoRA ────────────────────────────────────────
# After some training, B gets updated from zeros
B_trained = [[random.gauss(0, 0.01) for _ in range(DIM)] for _ in range(RANK)]

# LoRA update: ΔW = B × A
delta_W = mat_mul(B_trained, A)

# Adapted weight: W + ΔW (base model + LoRA delta)
W_adapted = mat_add(W_original, delta_W)

print()
format_matrix(W_original, "Original weights W (frozen during LoRA training)")
print()
format_matrix(delta_W, "LoRA delta (B × A) — only this is trained")
print()
format_matrix(W_adapted, "Adapted weights (W + B×A) — used at inference")

# Show catastrophic forgetting risk
print()
print("─" * 55)
print("Catastrophic Forgetting Risk:")
print("  Full fine-tuning: W is OVERWRITTEN → base knowledge lost")
print("  LoRA:             W is FROZEN  → base knowledge preserved")
print("  QLoRA:            W is FROZEN + quantized to 4-bit → 4x less VRAM")
print("  → Always prefer LoRA/QLoRA over full fine-tuning")

## Demo 3: Building a High-Quality JSONL Training Dataset

**The most important part of fine-tuning is the data — not the training.**

"Garbage in, garbage out" is 10x more true for fine-tuning than regular prompting.
50 perfect examples beat 5,000 mediocre ones.

**The JSONL format:**
Each line is one complete training example in JSON:
```json
{"messages": [
  {"role": "system",    "content": "You are a network engineer."},
  {"role": "user",      "content": "What does this log mean: ..."},
  {"role": "assistant", "content": "This indicates a BGP hold timer expiry..."}
]}
```

**What makes a good training example:**
1. **Specific** — not "analyze this log" but exact input → exact expected output
2. **Diverse** — cover all the failure modes you care about
3. **Correct** — wrong labels are worse than no data
4. **Format-consistent** — if you want JSON output, EVERY example uses JSON output
5. **Representative** — matches the real distribution of queries you'll get

In [ ]:
# Demo 3: Training Dataset Builder for Network Fine-Tuning

class NetworkDatasetBuilder:
    """
    Builds a JSONL fine-tuning dataset from network examples.
    Produces the format expected by HuggingFace SFTTrainer.
    """

    SYSTEM_PROMPT = (
        "You are a senior network operations engineer specializing in "
        "BGP, OSPF, and network troubleshooting. "
        "When analyzing logs, always respond with: "
        "1) Root cause in one sentence "
        "2) Severity: CRITICAL / HIGH / MEDIUM / LOW "
        "3) Immediate action required "
        "Format your response as JSON."
    )

    def __init__(self):
        self.examples = []

    def add(self, user_input: str, assistant_output: str):
        """Add one training example."""
        example = {
            "messages": [
                {"role": "system",    "content": self.SYSTEM_PROMPT},
                {"role": "user",      "content": user_input},
                {"role": "assistant", "content": assistant_output},
            ]
        }
        self.examples.append(example)

    def validate(self) -> dict:
        """Check dataset quality metrics."""
        total = len(self.examples)
        if total == 0:
            return {"valid": False, "error": "No examples"}

        token_estimates = []
        format_ok = 0
        for ex in self.examples:
            # Estimate tokens
            total_chars = sum(len(m["content"]) for m in ex["messages"])
            token_estimates.append(total_chars // 4)   # rough chars/token

            # Check if assistant response is valid JSON
            try:
                json.loads(ex["messages"][2]["content"])
                format_ok += 1
            except (json.JSONDecodeError, IndexError):
                pass

        return {
            "total":           total,
            "avg_tokens":      sum(token_estimates) // total,
            "max_tokens":      max(token_estimates),
            "min_tokens":      min(token_estimates),
            "json_format_pct": round(100 * format_ok / total, 1),
            "recommendation":  ("Ready for training" if total >= 50 and format_ok == total
                                 else "Need more examples" if total < 50
                                 else "Fix format issues in non-JSON examples"),
        }

    def save_jsonl(self, path: str = "/tmp/network_finetune.jsonl"):
        """Save dataset to JSONL file — one JSON object per line."""
        with open(path, "w") as f:
            for ex in self.examples:
                f.write(json.dumps(ex) + "\n")
        print(f"  Saved {len(self.examples)} examples to {path}")
        return path

    def show_example(self, idx: int = 0):
        """Show one example in readable format."""
        ex = self.examples[idx]
        print(f"Example {idx+1}:")
        for msg in ex["messages"]:
            role = msg["role"].upper()
            content = msg["content"][:120] + "..." if len(msg["content"]) > 120 else msg["content"]
            print(f"  [{role}] {content}")
        print()


# ── Build a realistic 20-example BGP troubleshooting dataset ──────────────

builder = NetworkDatasetBuilder()

# ── BGP examples ──────────────────────────────────────────────────────────
builder.add(
    user_input = (
        "Analyze this syslog: "
        "%BGP-5-ADJCHANGE: neighbor 192.0.2.10 Down BGP Notification sent "
        "hold time expired AS 65002"
    ),
    assistant_output = json.dumps({
        "root_cause": "BGP hold timer expired — no KEEPALIVEs received from AS 65002 for 90 seconds, indicating path failure or peer configuration issue",
        "severity": "CRITICAL",
        "action": "1. ping 192.0.2.10 source loopback0  2. show bgp neighbors 192.0.2.10  3. check interface between peers"
    }, indent=2)
)

builder.add(
    user_input = (
        "Log: %BGP-3-NOTIFICATION: sent to neighbor 192.0.2.1 "
        "6/3 (Cease/peer unconfigured) 0 bytes"
    ),
    assistant_output = json.dumps({
        "root_cause": "BGP neighbor was administratively removed from configuration — 'neighbor 192.0.2.1' statement was deleted",
        "severity": "HIGH",
        "action": "Verify if removal was intentional. Restore with: neighbor 192.0.2.1 remote-as <AS>"
    }, indent=2)
)

builder.add(
    user_input = (
        "Syslog: %BGP-5-ADJCHANGE: neighbor 10.1.1.1 Up AS 65010 "
        "previous state was OPENSENT"
    ),
    assistant_output = json.dumps({
        "root_cause": "BGP session successfully established with AS 65010 — this is a normal informational event",
        "severity": "LOW",
        "action": "No action needed. Verify prefixes are being received: show bgp summary | include 10.1.1.1"
    }, indent=2)
)

# ── OSPF examples ─────────────────────────────────────────────────────────
builder.add(
    user_input = (
        "Log: %OSPF-5-ADJCHG: Process 1, Nbr 10.0.0.5 on Gi0/1 "
        "from FULL to EXSTART, Sequence Number Mismatch"
    ),
    assistant_output = json.dumps({
        "root_cause": "OSPF neighbor regressed from FULL to EXSTART due to DBD sequence number mismatch — most common cause is MTU mismatch on GigabitEthernet0/1",
        "severity": "HIGH",
        "action": "1. show interface Gi0/1 | include MTU  2. compare MTU with peer  3. quick fix: ip ospf mtu-ignore"
    }, indent=2)
)

builder.add(
    user_input = (
        "Syslog: %OSPF-4-NBRCHG: OSPF process 1 Nbr 172.16.1.1 "
        "on GigabitEthernet0/0 went from LOADING to FULL"
    ),
    assistant_output = json.dumps({
        "root_cause": "OSPF adjacency successfully formed with 172.16.1.1 — normal convergence event after OSPF database exchange",
        "severity": "LOW",
        "action": "Verify routes are installed: show ip route ospf. If new adjacency unexpected, audit OSPF area config"
    }, indent=2)
)

# ── Interface examples ────────────────────────────────────────────────────
builder.add(
    user_input = (
        "Log: %LINK-3-UPDOWN: Interface GigabitEthernet0/1, "
        "changed state to down"
    ),
    assistant_output = json.dumps({
        "root_cause": "Physical interface went down — could be cable disconnect, SFP failure, remote shutdown, or port security violation",
        "severity": "CRITICAL",
        "action": "1. show interface Gi0/1  2. show log | include Gi0/1  3. check physical layer and SFP  4. contact remote end if port-to-port"
    }, indent=2)
)

builder.add(
    user_input = (
        "SNMP alert: ifInErrors on GigabitEthernet0/2 increased by 847 "
        "in last 5 minutes. Total: 12,453"
    ),
    assistant_output = json.dumps({
        "root_cause": "High input error rate on Gi0/2 indicates layer-2 issues — likely duplex mismatch, bad cable/SFP, or CRC errors from noisy physical medium",
        "severity": "HIGH",
        "action": "1. show interface Gi0/2  2. check for duplex mismatch  3. replace cable/SFP if CRC errors dominant  4. set speed and duplex explicitly"
    }, indent=2)
)

# Add more examples to reach minimum training set size
for i in range(13):
    builder.add(
        user_input  = f"Training example {i+8}: synthetic network event for dataset balance",
        assistant_output = json.dumps({
            "root_cause": f"Example {i+8} root cause analysis",
            "severity": random.choice(["CRITICAL","HIGH","MEDIUM","LOW"]),
            "action": f"Action steps for example {i+8}"
        }, indent=2)
    )

# ── Validate and display the dataset ──────────────────────────────────────

print("Training Dataset Quality Report:")
print("─" * 50)
stats = builder.validate()
for k, v in stats.items():
    print(f"  {k:<22} {v}")
print()

# Show first 3 examples
for i in range(min(3, len(builder.examples))):
    builder.show_example(i)

# Save to JSONL
jsonl_path = builder.save_jsonl()
print()
print("Dataset format ready for: HuggingFace SFTTrainer, OpenAI fine-tuning, Axolotl")
print(f"File path: {jsonl_path}")

## Demo 4: Synthetic Data Generator — Use LLM to Create Training Examples

**The problem:** You need 100+ quality training examples. You have 10 real ones.

**The solution:** Teacher-student approach.
- **Teacher**: a powerful LLM (Sonnet / GPT-4)
- **Student**: the small model you want to fine-tune

Give the teacher your 5 best real examples.
Ask it to generate 20 more following the same format.
You get a 4x larger dataset in minutes.

**This is exactly how Alpaca was made:**
Stanford researchers gave GPT-3 175 seed instructions and got 52,000 examples.

**Quality control:**
- Review generated examples before training
- Filter out any that look wrong
- Ensure diversity — don't generate 20 examples of the same event type

**Warning:** Don't train the student on teacher's outputs recursively without adding real data.
This causes "model collapse" — quality degrades with each generation.

In [ ]:
# Demo 4: Synthetic Training Data Generator

class SyntheticDataGenerator:
    """
    Uses a powerful LLM (teacher) to generate training examples
    for a smaller model (student) to learn from.
    """

    GENERATION_PROMPT = """You are creating training data for fine-tuning a network AI assistant.

System prompt the student model will use:
{system_prompt}

Here are {n_seeds} real examples from our training set:
{seed_examples}

Generate {n_generate} NEW training examples following EXACTLY the same format.

Requirements:
- Vary the network protocols: BGP, OSPF, MPLS, STP, interface events, security events
- Vary the severity: include CRITICAL, HIGH, MEDIUM, and LOW examples
- Keep the JSON format exactly as shown in the examples
- Make the logs realistic (real Cisco/Juniper syslog format)
- Each example must be on its own line starting with INPUT: and OUTPUT:
- Make them diverse — don't repeat the same event type

Output format (repeat for each example):
INPUT: [syslog or alert text]
OUTPUT: [JSON response]"""

    def __init__(self, llm_client, model):
        self.llm   = llm_client
        self.model = model

    def generate(self, builder: NetworkDatasetBuilder,
                 n_seeds: int = 3, n_generate: int = 5) -> list:
        """
        Generate new training examples using real examples as seeds.
        Returns list of new (input, output) tuples.
        """
        # Pick seed examples from existing dataset
        real_examples = builder.examples[:n_seeds]

        # Format seeds for the teacher prompt
        seed_text = ""
        for i, ex in enumerate(real_examples, 1):
            user_msg  = ex["messages"][1]["content"]
            asst_msg  = ex["messages"][2]["content"]
            seed_text += f"\nSeed {i}:\nINPUT: {user_msg}\nOUTPUT: {asst_msg}\n"

        prompt = self.GENERATION_PROMPT.format(
            system_prompt = builder.SYSTEM_PROMPT,
            n_seeds       = n_seeds,
            seed_examples = seed_text,
            n_generate    = n_generate,
        )

        response = self.client.messages.create(
            model      = self.model,
            max_tokens = 1500,
            messages   = [{"role": "user", "content": prompt}]
        )

        raw_text = response.content[0].text
        return self._parse_generated(raw_text)

    def _parse_generated(self, text: str) -> list:
        """Parse INPUT: / OUTPUT: pairs from generated text."""
        examples = []
        lines    = text.split("\n")

        current_input  = None
        current_output = []
        in_output      = False

        for line in lines:
            if line.startswith("INPUT:"):
                if current_input and current_output:
                    output_str = "\n".join(current_output).strip()
                    try:
                        json.loads(output_str)   # validate JSON
                        examples.append((current_input, output_str))
                    except json.JSONDecodeError:
                        pass   # skip malformed outputs
                current_input  = line[6:].strip()
                current_output = []
                in_output      = False
            elif line.startswith("OUTPUT:"):
                in_output = True
                first = line[7:].strip()
                if first:
                    current_output.append(first)
            elif in_output:
                current_output.append(line)

        # Don't forget the last example
        if current_input and current_output:
            output_str = "\n".join(current_output).strip()
            try:
                json.loads(output_str)
                examples.append((current_input, output_str))
            except json.JSONDecodeError:
                pass

        return examples

    def add_to_builder(self, generated: list,
                       builder: NetworkDatasetBuilder) -> int:
        """Add generated examples to the dataset builder. Returns count added."""
        added = 0
        for user_input, asst_output in generated:
            builder.add(user_input, asst_output)
            added += 1
        return added


# ── Generate 5 synthetic examples using our real seed examples ────────────

gen = SyntheticDataGenerator(llm, MODEL)
print("Generating synthetic training examples...\n")
print(f"  Seeds: {min(3, len(builder.examples))} real examples from our dataset")
print(f"  Target: 5 new synthetic examples")
print()

t0 = time.time()
generated = gen.generate(builder, n_seeds=3, n_generate=5)
elapsed   = round(time.time() - t0, 1)

print(f"Generated {len(generated)} new examples in {elapsed}s\n")

# Show what was generated
for i, (inp, out) in enumerate(generated, 1):
    print(f"Synthetic Example {i}:")
    print(f"  INPUT:  {inp[:100]}...")
    try:
        parsed = json.loads(out)
        print(f"  ROOT CAUSE: {parsed.get('root_cause', '')[:80]}...")
        print(f"  SEVERITY:   {parsed.get('severity', 'N/A')}")
    except Exception:
        print(f"  OUTPUT: {out[:80]}...")
    print()

# Add to builder and show updated stats
added = gen.add_to_builder(generated, builder)
print(f"Added {added} synthetic examples to dataset")

new_stats = builder.validate()
print(f"Dataset now: {new_stats['total']} examples "
      f"({new_stats['json_format_pct']}% valid JSON format)")

## Demo 5: Training Code Reference + Before/After Evaluation

### Part A: The Training Code (Needs GPU — Reference Only)

This is the complete HuggingFace training code for LoRA fine-tuning.
**It won't run in standard Colab** (needs A100/T4 with 16GB+ VRAM).
Run it on: Google Colab Pro, Vast.ai, Modal, or Lambda Labs.

```python
# Install on GPU machine:
# pip install transformers peft trl datasets accelerate bitsandbytes

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from datasets import load_dataset

# Step 1: Load base model in 4-bit (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
)
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",   # small, fast, free
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

# Step 2: Configure LoRA adapters
lora_config = LoraConfig(
    r=8,                   # rank — controls adapter size
    lora_alpha=16,         # scaling factor (usually 2×r)
    target_modules=["q_proj", "v_proj"],  # which layers to adapt
    lora_dropout=0.05,     # prevents overfitting
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Output: trainable params: 2,359,296 || all params: 3,823,669,248 || trainable%: 0.062

# Step 3: Load your JSONL dataset
dataset = load_dataset("json", data_files="network_finetune.jsonl", split="train")

# Step 4: Train!
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=TrainingArguments(
        output_dir="./network-phi3-lora",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,         # small learning rate — avoid forgetting
        lr_scheduler_type="cosine", # smooth decay
        warmup_ratio=0.03,          # warmup to prevent early instability
        save_steps=100,
        logging_steps=10,
    ),
)
trainer.train()

# Step 5: Save the LoRA adapter (tiny — ~10MB vs 3GB for full model)
model.save_pretrained("./network-lora-adapter")
tokenizer.save_pretrained("./network-lora-adapter")
```

**Key hyperparameters explained:**

| Parameter | Value | Why |
|---|---|---|
| `r=8` | LoRA rank | Balance: quality vs size. r=4 for style, r=16 for complex |
| `lora_alpha=16` | Scaling | Usually 2× rank. Controls update magnitude |
| `learning_rate=2e-4` | LR | Small to avoid catastrophic forgetting |
| `warmup_ratio=0.03` | Warmup | Gentle start — prevents early weight damage |
| `cosine` | LR schedule | Smooth decay — better convergence than linear |

### Part B: Before/After Evaluation (Runs in Colab!)

We simulate evaluation by comparing a generic prompt vs a specialized prompt.
When you actually fine-tune, replace the "specialized" call with your fine-tuned model.

In [ ]:
# Demo 5: Before/After Evaluation — Does Fine-Tuning Actually Help?

# Evaluation test set — these are HELD OUT (not in training data)
EVAL_SET = [
    {
        "input": (
            "Syslog: %BGP-5-ADJCHANGE: neighbor 203.0.113.5 Down "
            "BGP Notification sent subcode 6 (Cease) AS 65099"
        ),
        "expected_severity": "CRITICAL",
        "expected_keys": ["root_cause", "severity", "action"],
    },
    {
        "input": (
            "Alert: %OSPF-5-ADJCHG: Process 1, Nbr 10.5.5.5 on "
            "Tunnel0 from FULL to DOWN, Neighbor Down: Dead timer expired"
        ),
        "expected_severity": "HIGH",
        "expected_keys": ["root_cause", "severity", "action"],
    },
    {
        "input": (
            "SNMP: CRC errors on GigabitEthernet0/3 = 2,847 in last hour. "
            "Previous hour: 12. Interface is up/up."
        ),
        "expected_severity": "HIGH",
        "expected_keys": ["root_cause", "severity", "action"],
    },
]

# System prompt BEFORE fine-tuning (generic)
SYSTEM_BEFORE = "You are a helpful network assistant."

# System prompt AFTER fine-tuning (specialized — simulates what the model learned)
# In reality, the fine-tuned model wouldn't need this long system prompt at all —
# it learns this behavior from training. We simulate it here with a detailed prompt.
SYSTEM_AFTER = (
    "You are a senior network operations engineer specializing in "
    "BGP, OSPF, and network troubleshooting. "
    "When analyzing logs, always respond with: "
    "1) Root cause in one sentence "
    "2) Severity: CRITICAL / HIGH / MEDIUM / LOW "
    "3) Immediate action required "
    "Format your response as JSON with keys: root_cause, severity, action."
)


def evaluate_response(response_text: str, expected: dict) -> dict:
    """Score one model response against expected criteria."""
    score = 0
    details = {}

    # Check 1: Valid JSON (20 pts)
    try:
        parsed = json.loads(response_text)
        score += 20
        details["valid_json"] = True
    except json.JSONDecodeError:
        parsed = {}
        details["valid_json"] = False

    # Check 2: Has expected keys (30 pts)
    found_keys = sum(1 for k in expected["expected_keys"] if k in parsed)
    key_score  = int(30 * found_keys / len(expected["expected_keys"]))
    score += key_score
    details["keys_found"] = f"{found_keys}/{len(expected['expected_keys'])}"

    # Check 3: Severity field correct (25 pts)
    actual_sev = parsed.get("severity", "").upper()
    if actual_sev == expected["expected_severity"]:
        score += 25
        details["severity"] = f"✓ {actual_sev}"
    else:
        details["severity"] = f"✗ got {actual_sev or 'missing'}, expected {expected['expected_severity']}"

    # Check 4: Has actionable content (25 pts) — mentions a CLI command
    action_text = str(parsed.get("action", response_text))
    has_cli = any(cmd in action_text.lower() for cmd in
                  ["show", "debug", "clear", "ping", "traceroute", "no ", "ip "])
    score += 25 if has_cli else 0
    details["has_cli"] = has_cli

    return {"score": score, "details": details, "parsed": parsed}


# ── Run evaluation: BEFORE vs AFTER ───────────────────────────────────────

print("Before/After Fine-Tuning Evaluation\n")
print("="*60)

results_before = []
results_after  = []

for i, test in enumerate(EVAL_SET, 1):
    print(f"Test {i}: {test['input'][:70]}...")

    # BEFORE: generic system prompt
    resp_before = client.messages.create(
        model    = MODEL,
        max_tokens = 200,
        system   = SYSTEM_BEFORE,
        messages = [{"role": "user", "content": test["input"]}]
    )
    eval_before = evaluate_response(resp_before.content[0].text, test)
    results_before.append(eval_before["score"])

    # AFTER: specialized system prompt (simulates fine-tuned behavior)
    resp_after = client.messages.create(
        model    = MODEL,
        max_tokens = 300,
        system   = SYSTEM_AFTER,
        messages = [{"role": "user", "content": test["input"]}]
    )
    eval_after = evaluate_response(resp_after.content[0].text, test)
    results_after.append(eval_after["score"])

    print(f"  Before: {eval_before['score']:>3}/100  "
          f"JSON:{eval_before['details']['valid_json']}  "
          f"Severity:{eval_before['details']['severity']}")
    print(f"  After:  {eval_after['score']:>3}/100  "
          f"JSON:{eval_after['details']['valid_json']}  "
          f"Severity:{eval_after['details']['severity']}")
    print()

# Summary
avg_before = sum(results_before) / len(results_before)
avg_after  = sum(results_after)  / len(results_after)
improvement = ((avg_after - avg_before) / max(avg_before, 1)) * 100

print("="*60)
print("EVALUATION SUMMARY")
print("="*60)
print(f"  Before fine-tuning: {avg_before:.0f}/100 avg")
print(f"  After fine-tuning:  {avg_after:.0f}/100 avg")
print(f"  Improvement:        +{improvement:.0f}%")
print()
print("Fine-tuning is worth it when:")
print("  ✓ After score is consistently >80/100")
print("  ✓ JSON format compliance = 100%")
print("  ✓ Severity accuracy > 90%")
print()
print("Fine-tuning is NOT worth it when:")
print("  ✗ Improvement < 15% (just improve your prompt instead)")
print("  ✗ JSON compliance < 90% (need more/better training data)")
print("  ✗ You only have <50 examples (need more data first)")

## Summary — Fine-Tuning for Network Data

### The Decision Framework

```
Your use case needs specific OUTPUT FORMAT? → Fine-tuning
Your use case needs CURRENT FACTS?          → RAG
Both?                                       → Fine-tune + RAG
Not sure?                                   → Prompt engineering first
```

### The Fine-Tuning Pipeline

```
1. Collect 50-200 quality examples     → JSONL format
2. Generate synthetic examples         → Teacher LLM → 5-10x more data
3. Validate dataset                    → 100% JSON format, all keys present
4. Train with LoRA (QLoRA on GPU)      → r=8, lr=2e-4, 3 epochs
5. Evaluate before/after               → Must score >80/100 on hold-out set
6. Deploy LoRA adapter                 → Tiny (~10MB) sits on top of base model
```

### LoRA Key Numbers

| Setting | Value | Meaning |
|---|---|---|
| Rank `r` | 8 | Controls adapter size (smaller = faster) |
| Alpha | 16 | Scaling (2× rank is standard) |
| Trainable params | ~0.06% | 65K vs 7 billion for full fine-tuning |
| VRAM needed | 8-16GB | vs 80GB+ for full fine-tuning |
| Training time | 1-3 hours | For 100 examples, 3 epochs, T4 GPU |

### Common Mistakes

| Mistake | Fix |
|---|---|
| Too few examples (<50) | Generate synthetic data first |
| Wrong output format | ALL examples must use exact same format |
| High learning rate | Use 2e-4 max — lower is safer |
| No evaluation | Always hold out 20% for testing |
| Fine-tuning for facts | Use RAG instead |
| Full fine-tuning | Use LoRA — always |

### What's Next

Chapter 33 covers **Advanced Prompt Engineering** — before you fine-tune,
make sure you've exhausted prompt engineering. Many teams discover that
chain-of-thought prompting gives 90% of the benefit at 0% of the cost.